# RAMSES free-GPU measurement notebook

This notebook collects **supplemental smoke-test measurements only**. It does not implement RAMSES, GPUDirect Storage, whole-node energy, PLC/TSN, or reproduce the A100 results. Use a Colab GPU runtime; the notebook fails closed if CUDA is unavailable. Output conforms to `code/measurement-schema.json`.


In [ ]:
!pip -q install "transformers==4.48.3" "pynvml==12.0.0"


In [ ]:
import json, os, platform, statistics, subprocess, time, uuid
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU; do not report CPU results as GPU results"
print(subprocess.check_output(["nvidia-smi","--query-gpu=name,driver_version,memory.total","--format=csv,noheader"],text=True))
print({"torch":torch.__version__,"cuda":torch.version.cuda,"transformers":__import__('transformers').__version__,"python":platform.python_version()})


In [ ]:
MODEL="sshleifer/tiny-gpt2"; PRECISION="fp16"; RUNS=5; REQUESTS=100; INPUT_TOKENS=32; OUTPUT_TOKENS=16
tok=AutoTokenizer.from_pretrained(MODEL); tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype=torch.float16).cuda().eval()
prompt="Inspect the vibration signal and summarize the anomaly."
ids=tok(prompt,return_tensors="pt").input_ids.cuda()
if ids.shape[1]<INPUT_TOKENS: ids=ids.repeat(1,(INPUT_TOKENS+ids.shape[1]-1)//ids.shape[1])[:,:INPUT_TOKENS]
else: ids=ids[:,:INPUT_TOKENS]


In [ ]:
records=[]
def sync_time(fn):
 torch.cuda.synchronize(); start=time.perf_counter_ns(); out=fn(); torch.cuda.synchronize(); return out,(time.perf_counter_ns()-start)/1e6
def emit(task,ms,run,out_tokens):
 records.append(dict(run_id=f"colab-{run}",system="pytorch-smoke",task=task,latency_ms=ms,input_tokens=int(ids.shape[1]),output_tokens=out_tokens,batch=1,concurrency=1,request_count=1,precision=PRECISION,model=MODEL))
with torch.inference_mode():
 for run in range(RUNS):
  for _ in range(10): model(ids)
  for _ in range(REQUESTS):
   _,ms=sync_time(lambda:model(ids)); emit("scoring",ms,run,0)
   past=model(ids,use_cache=True).past_key_values
   _,ms=sync_time(lambda:model(ids[:,-1:],past_key_values=past,use_cache=True)); emit("continuation",ms,run,1)
   _,ms=sync_time(lambda:model.generate(ids,max_new_tokens=1,do_sample=False,pad_token_id=tok.eos_token_id)); emit("ttft",ms,run,1)
   _,ms=sync_time(lambda:model.generate(ids,max_new_tokens=OUTPUT_TOKENS,do_sample=False,pad_token_id=tok.eos_token_id)); emit("generation",ms,run,OUTPUT_TOKENS)
open("colab_raw.jsonl","w").write("".join(json.dumps(r)+"\n" for r in records))
print(len(records),"measured records written")


In [ ]:
def pct(xs,p):
 xs=sorted(xs); k=(len(xs)-1)*p/100; a=int(k); b=min(a+1,len(xs)-1); return xs[a]*(b-k)+xs[b]*(k-a)
for task in ("scoring","continuation","ttft","generation"):
 xs=[r["latency_ms"] for r in records if r["task"]==task]
 print(task,{p:round(pct(xs,p),4) for p in (50,95,99,99.9,100)})
from google.colab import files
files.download("colab_raw.jsonl")
